# 03 - Analisis Cost-Sensitive Classification (Bonus +2 Poin)
**Penanggung Jawab:** Anggota 5  
**Kelompok:** G15 - MessiLearning (IF3170 Inteligensi Artifisial)

---
### Formulasi Matematis Ambang Batas Optimal (*Bayesian Decision Theory*)
- **False Positive ($FP$)**: Model memprediksi pesanan batal ($\hat{y}=1$), padahal tamu hadir ($y=0$). Menyebabkan risiko **overbooking** dengan biaya $C(FP) = 2$.
- **False Negative ($FN$)**: Model memprediksi tamu hadir ($\hat{y}=0$), padahal pesanan batal ($y=1$). Kamar dibiarkan kosong dengan biaya $C(FN) = 1$.

Prediksi $\hat{y}=1$ dipilih jika biaya ekspektasian untuk kelas 1 lebih kecil atau sama dengan kelas 0. Pada kasus seri, kedua keputusan memiliki biaya yang sama.
$$(1 - P(y=1|x)) \cdot C(FP) \leq P(y=1|x) \cdot C(FN)$$
$$\tau^* = \frac{C(FP)}{C(FP) + C(FN)} = \frac{2}{2 + 1} = \frac{2}{3} \approx 0.6667$$

In [ ]:
# === Bagian Anggota 5: Muat Probabilitas Holdout dari Notebook Utama ===
import sys
from pathlib import Path

import pandas as pd

cost_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(cost_root))

from src.utils import evaluate_cost_thresholds

cost_id, cost_target = "id", "is_canceled"
cost_path = cost_root / "output" / "validation_cost_scores.csv"
if not cost_path.is_file():
    raise FileNotFoundError(
        "Jalankan notebook template utama sampai bagian Cost-Sensitive Classification "
        "untuk menghasilkan output/validation_cost_scores.csv."
    )
cost_scores = pd.read_csv(cost_path, dtype={cost_id: str}, float_precision="round_trip")
cost_validation = pd.read_csv(cost_root / "data" / "processed" / "val_dtl.csv", dtype={cost_id: str})
pd.testing.assert_frame_equal(
    cost_scores[[cost_id, cost_target]], cost_validation[[cost_id, cost_target]]
)

## Komparasi Total Biaya pada Lima Model

Jalankan notebook template utama sampai bagian **Cost-Sensitive Classification** untuk menyiapkan probabilitas kelas 1 pada split validasi. Notebook ini membandingkan ambang 0,50 dan 2/3 menggunakan probabilitas tersebut. Pada nilai tepat di ambang, kelas 1 dipilih. Untuk SVM, Platt scaling dipelajari dari skor *out-of-fold* data latih.

In [ ]:
cost_models = [
    name for name in cost_scores.columns if name not in (cost_id, cost_target)
]
cost_results = pd.DataFrame([
    {"Model": name, **row}
    for name in cost_models
    for row in evaluate_cost_thresholds(cost_scores[cost_target].to_numpy(), cost_scores[name].to_numpy())
])
display(cost_results.round(4))

base_cost = cost_results[cost_results["Aturan"] == "Standar"].set_index("Model")
adjusted_cost = cost_results[cost_results["Aturan"] == "Cost-sensitive"].set_index("Model")
cost_columns = ["FP", "FN", "Macro F1", "Total biaya"]
cost_changes = adjusted_cost[cost_columns] - base_cost[cost_columns]
display(cost_changes.add_prefix("Δ ").round(4))
print("Δ dihitung dari hasil cost-sensitive dikurangi hasil standar. Δ biaya negatif berarti penalti berkurang.")